In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType,DateType
)

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [15]:
customers = spark.read.csv(
    "s3a://ment4/customers.csv",
    header=True,
    inferSchema=True
)

transactions = spark.read.csv(
    "s3a://ment4/transactions.csv",
    header=True,
    inferSchema=True
)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3a://ment4/customers.csv.

In [2]:
transactions_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("status", StringType(), True)
])

In [3]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])

In [4]:
transactions_df = spark.read \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(transactions_schema) \
    .csv("s3a://ment4/transactions.csv")

In [5]:
cust_df = spark.read \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(customers_schema) \
    .csv("s3a://ment4/cust.csv")

In [6]:
transactions_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- status: string (nullable = true)



In [7]:
cust_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- segment: string (nullable = true)



In [8]:
cust_df.createOrReplaceTempView("customers")
transactions_df.createOrReplaceTempView("transactions")

In [10]:
city_null_cnt = spark.sql("""
    SELECT COUNT(*) AS null_cnt
    FROM customers
    WHERE city IS NULL
""")
city_null_cnt.show()

+--------+
|null_cnt|
+--------+
|       2|
+--------+



In [11]:
amount_null_cnt = spark.sql("""
    SELECT COUNT(*) AS null_cnt
    FROM transactions
    WHERE amount IS NULL
""")

amount_null_cnt.show()

+--------+
|null_cnt|
+--------+
|       3|
+--------+



In [12]:
transactions_clean = spark.sql("""
    SELECT *
    FROM transactions
    WHERE amount IS NOT NULL
""")

transactions_clean.show()

+--------------+-----------+----------+-------+----------------+---------+
|transaction_id|customer_id|   product| amount|transaction_date|   status|
+--------------+-----------+----------+-------+----------------+---------+
|         T0001|       C013|     Chair| 113.29|      2024-08-21| Refunded|
|         T0002|       C011|  Keyboard| 988.33|      2024-04-11|Cancelled|
|         T0003|       C015|   Monitor| 408.35|      2024-03-04|  Pending|
|         T0004|       C018|     Chair|1124.32|      2024-04-19|Cancelled|
|         T0005|       C013|      Desk| 340.69|      2024-02-05|Completed|
|         T0006|       C003|    Laptop|1293.74|      2024-02-09|Completed|
|         T0007|       C014|    Webcam| 109.34|      2024-04-07|Completed|
|         T0008|       C015|   Printer| 388.35|      2024-05-21|Cancelled|
|         T0009|       C001|     Phone|1027.34|      2024-05-17|  Pending|
|         T0010|       C011|     Phone| 450.85|      2024-02-10|Completed|
|         T0011|       C0

In [14]:
customers_clean = spark.sql("""
    SELECT
        customer_id,
        name,
        COALESCE(city, 'Unknown') AS city
    FROM customers
""")

customers_clean.show()

+-----------+----------------+----------+
|customer_id|            name|      city|
+-----------+----------------+----------+
|       C001|    Leyla Aliyev|   Shirvan|
|       C002|  Nigar Huseynov|   Shirvan|
|       C003| Nargiz Mammadov|  Lankaran|
|       C004|  Aylin Mammadov|     Ganja|
|       C005|   Khumar Aliyev|  Lankaran|
|       C006|  Nargiz Abbasov|     Ganja|
|       C007|   Rashad Aliyev|   Unknown|
|       C008|  Zeynab Quliyev|   Sumgait|
|       C009| Orkhan Mammadov|      Baku|
|       C010|   Aysel Quliyev|  Lankaran|
|       C011|   Murad Karimov|      Baku|
|       C012|Nargiz Ismayilov|Nakhchivan|
|       C013|   Aysel Bagirov|     Ganja|
|       C014|    Elvin Rzayev|Nakhchivan|
|       C015|  Nigar Mammadov|   Unknown|
|       C016|  Aysel Huseynov|   Sumgait|
|       C017| Rashad Mammadov|  Lankaran|
|       C018|   Nargiz Rzayev|     Ganja|
|       C019|  Rashad Karimov|     Ganja|
|       C020|    Elvin Rzayev|Nakhchivan|
+-----------+----------------+----